import libraries

In [1]:
import pandas as pd
import glob
import os

print("Libraries loaded successfully")

Libraries loaded successfully


 load all LFS monthly files:

In [2]:
# Only grab files that start with 'pub' to exclude codebook/layout files
lfs_files = glob.glob('../data/raw/202*/**/*.csv', recursive=True)
lfs_files = [f for f in lfs_files if os.path.basename(f).startswith('pub')]

print(f"Found {len(lfs_files)} LFS files:")
for f in lfs_files:
    print(os.path.basename(f))

Found 64 LFS files:
pub0121.csv
pub0221.csv
pub0321.csv
pub0421.csv
pub0521.csv
pub0621.csv
pub0721.csv
pub0821.csv
pub0921.csv
pub1021.csv
pub1121.csv
pub1221.csv
pub0122.csv
pub0222.csv
pub0322.csv
pub0422.csv
pub0522.csv
pub0622.csv
pub0722.csv
pub0822.csv
pub0922.csv
pub1022.csv
pub1122.csv
pub1222.csv
pub0123.csv
pub0223.csv
pub0323.csv
pub0423.csv
pub0523.csv
pub0623.csv
pub0723.csv
pub0823.csv
pub0923.csv
pub1023.csv
pub1123.csv
pub1223.csv
pub0124.csv
pub0224.csv
pub0324.csv
pub0424.csv
pub0524.csv
pub0624.csv
pub0724.csv
pub0824.csv
pub0924.csv
pub1024.csv
pub1124.csv
pub1224.csv
pub0125.csv
pub0225.csv
pub0325.csv
pub0425.csv
pub0525.csv
pub0625.csv
pub0725.csv
pub0825.csv
pub0925.csv
pub1025.csv
pub1125.csv
pub1225.csv
pub0126.csv
pub0226.csv
pub0326.csv
pub0426.csv


combine all LFS files into one DataFrame:

In [9]:
import gc
import os

COLS_TO_KEEP = [
    'PROV', 'AGE_12', 'SEX', 'MARSTAT', 'EDUC',
    'IMMIG', 'NAICS_21', 'NOC_10', 'LFSSTAT',
    'FTPTMAIN', 'HRLYEARN', 'UHRSMAIN', 'TENURE',
    'PERMTEMP', 'COWMAIN'
]

# Create a folder for individual parquet files
os.makedirs('../data/processed/lfs_parts', exist_ok=True)

for file in lfs_files:
    filename = os.path.basename(file)
    month = int(filename[3:5])
    year = int('20' + filename[5:7])
    
    out_file = f'../data/processed/lfs_parts/{filename.replace(".csv", ".parquet")}'
    
    # Skip if already processed
    if os.path.exists(out_file):
        print(f"Already done: {filename}")
        continue
    
    df = pd.read_csv(
        file,
        encoding='latin-1',
        usecols=lambda c: c in COLS_TO_KEEP,
        low_memory=True
    )
    
    df['year'] = year
    df['month'] = month
    df['date'] = pd.to_datetime({'year': df['year'], 'month': df['month'], 'day': 1})
    
    for col in df.select_dtypes('float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes('int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    
    df.to_parquet(out_file, index=False)
    del df
    gc.collect()
    print(f"Saved: {filename}")

print("\nAll files processed!")

Saved: pub0121.csv
Saved: pub0221.csv
Saved: pub0321.csv
Saved: pub0421.csv
Saved: pub0521.csv
Saved: pub0621.csv
Saved: pub0721.csv
Saved: pub0821.csv
Saved: pub0921.csv
Saved: pub1021.csv
Saved: pub1121.csv
Saved: pub1221.csv
Saved: pub0122.csv
Saved: pub0222.csv
Saved: pub0322.csv
Saved: pub0422.csv
Saved: pub0522.csv
Saved: pub0622.csv
Saved: pub0722.csv
Saved: pub0822.csv
Saved: pub0922.csv
Saved: pub1022.csv
Saved: pub1122.csv
Saved: pub1222.csv
Saved: pub0123.csv
Saved: pub0223.csv
Saved: pub0323.csv
Saved: pub0423.csv
Saved: pub0523.csv
Saved: pub0623.csv
Saved: pub0723.csv
Saved: pub0823.csv
Saved: pub0923.csv
Saved: pub1023.csv
Saved: pub1123.csv
Saved: pub1223.csv
Saved: pub0124.csv
Saved: pub0224.csv
Saved: pub0324.csv
Saved: pub0424.csv
Saved: pub0524.csv
Saved: pub0624.csv
Saved: pub0724.csv
Saved: pub0824.csv
Saved: pub0924.csv
Saved: pub1024.csv
Saved: pub1124.csv
Saved: pub1224.csv
Saved: pub0125.csv
Saved: pub0225.csv
Saved: pub0325.csv
Saved: pub0425.csv
Saved: pub05

In [13]:
import glob

# Load all individual parquet files and combine
part_files = glob.glob('../data/processed/lfs_parts/*.parquet')
print(f"Found {len(part_files)} part files, combining...")

lfs = pd.concat([pd.read_parquet(f) for f in sorted(part_files)], ignore_index=True)

print(f"\nTotal rows: {len(lfs):,}")
print(f"Columns: {list(lfs.columns)}")
print(f"Memory: {lfs.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Date range: {lfs['date'].min().date()} → {lfs['date'].max().date()}")

# Save final combined file
lfs.to_parquet('../data/processed/lfs_combined.parquet', index=False)
print("\nFinal combined file saved!")

Found 64 part files, combining...

Total rows: 6,759,638
Columns: ['LFSSTAT', 'PROV', 'AGE_12', 'MARSTAT', 'EDUC', 'COWMAIN', 'IMMIG', 'NAICS_21', 'NOC_10', 'UHRSMAIN', 'FTPTMAIN', 'TENURE', 'HRLYEARN', 'PERMTEMP', 'year', 'month', 'date']
Memory: 315.9 MB
Date range: 2021-01-01 → 2026-04-01

Final combined file saved!


 load CPI data:


In [14]:
cpi_raw = pd.read_csv(
    '../data/raw/cpi/1810000401-eng.csv',
    skiprows=9,
    header=0,
    encoding='latin-1',
    on_bad_lines='skip'
)

# Clean up
cpi_raw = cpi_raw.dropna(subset=[cpi_raw.columns[0]])
cpi_raw = cpi_raw[~cpi_raw.iloc[:,0].str.contains('2002', na=False)]

# Melt wide to long
cpi = cpi_raw.melt(
    id_vars=cpi_raw.columns[0],
    var_name='date_raw',
    value_name='cpi_value'
)

cpi.columns = ['product', 'date_raw', 'cpi_value']

# Convert "January 2021" format to datetime
cpi['date'] = pd.to_datetime(cpi['date_raw'], format='%B %Y')
cpi['cpi_value'] = pd.to_numeric(cpi['cpi_value'], errors='coerce')
cpi = cpi.dropna(subset=['cpi_value'])

# Keep only All-items CPI
cpi_allitems = cpi[cpi['product'].str.contains('All-items', na=False)].copy()

print(f"CPI rows: {len(cpi_allitems)}")
print(f"Date range: {cpi_allitems['date'].min().date()} → {cpi_allitems['date'].max().date()}")
print(cpi_allitems[['product','date','cpi_value']].head())

CPI rows: 192
Date range: 2021-01-01 → 2026-04-01
                                  product       date  cpi_value
0                               All-items 2021-01-01      138.2
10  All-items excluding food and energy 7 2021-01-01      132.8
11           All-items excluding energy 7 2021-01-01      136.7
27                              All-items 2021-02-01      138.9
37  All-items excluding food and energy 7 2021-02-01      133.1


 load wages data:

In [15]:
wages_raw = pd.read_csv(
    '../data/raw/wages/1410006301-eng.csv',
    skiprows=13,
    header=0,
    encoding='latin-1',
    on_bad_lines='skip'
)

wages_raw = wages_raw.dropna(subset=[wages_raw.columns[0]])

wages = wages_raw.melt(
    id_vars=wages_raw.columns[0],
    var_name='date_raw',
    value_name='avg_hourly_wage'
)

wages.columns = ['industry', 'date_raw', 'avg_hourly_wage']

# Fixed format to match "January 2021"
wages['date'] = pd.to_datetime(wages['date_raw'], format='%B %Y')
wages['avg_hourly_wage'] = pd.to_numeric(wages['avg_hourly_wage'], errors='coerce')
wages = wages.dropna(subset=['avg_hourly_wage'])

print(f"Wages rows: {len(wages)}")
print(f"Date range: {wages['date'].min().date()} → {wages['date'].max().date()}")
print(wages['industry'].unique())

Wages rows: 1235
Date range: 2021-01-01 → 2026-05-01
['Total employees, all industries 8' 'Goods-producing sector 9'
 'Agriculture 10'
 'Forestry, fishing, mining, quarrying, oil and gas 11 12' 'Utilities'
 'Construction' 'Manufacturing' 'Services-producing sector 13'
 'Wholesale and retail trade' 'Transportation and warehousing'
 'Finance, insurance, real estate, rental and leasing'
 'Professional, scientific and technical services'
 'Business, building and other support services 14' 'Educational services'
 'Health care and social assistance' 'Information, culture and recreation'
 'Accommodation and food services'
 'Other services (except public administration)' 'Public administration']


quick sanity check:

In [12]:
print("=== LFS DATA ===")
print(f"Rows: {len(lfs):,}")
print(f"Columns: {lfs.shape[1]}")
print(f"Date range: {lfs['date'].min().date()} → {lfs['date'].max().date()}")

print("\n=== CPI DATA ===")
print(f"Rows: {len(cpi_allitems)}")
print(f"Date range: {cpi_allitems['date'].min().date()} → {cpi_allitems['date'].max().date()}")

print("\n=== WAGES DATA ===")
print(f"Rows: {len(wages)}")
print(f"Industries: {wages['industry'].nunique()}")

=== LFS DATA ===


NameError: name 'lfs' is not defined

In [16]:
# Document exactly what we built
print("=== PIPELINE SUMMARY ===")
print(f"LFS files loaded: {len(lfs_files)}")
print(f"Date range: {lfs['date'].min().date()} → {lfs['date'].max().date()}")
print(f"Total individual records: {len(lfs):,}")
print(f"CPI months: {len(cpi_allitems)}")
print(f"Wage industries tracked: {wages['industry'].nunique()}")
print(f"Unique provinces in LFS: {lfs['PROV'].nunique()}")

=== PIPELINE SUMMARY ===
LFS files loaded: 64
Date range: 2021-01-01 → 2026-04-01
Total individual records: 6,759,638
CPI months: 192
Wage industries tracked: 19
Unique provinces in LFS: 10


In [17]:
# Save cleaned versions to processed folder
lfs.to_parquet('../data/processed/lfs_combined.parquet', index=False)
cpi_allitems.to_parquet('../data/processed/cpi_clean.parquet', index=False)
wages.to_parquet('../data/processed/wages_clean.parquet', index=False)

print("All datasets saved to processed folder successfully")

All datasets saved to processed folder successfully
